In [15]:
#import libraries
from chembl_webresource_client.new_client import new_client
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

In [17]:
# Choose your target and endpoints here
CHEMBL_TARGET = "CHEMBL217"
ENDPOINTS = ["Ki"]   # e.g., ["EC50"] if you prefer
RELATIONS = ["="]
THR_CLASS = 100
CONVERT_TO_LOG = False     # True for pKi/pKd style
RM_HIGH_STD = True

# File paths (edit to your VS Code workspace folders)
RAW_ALL_CSV = "CHEMBL217_raw.csv"
RAW_STRUCT_FILTERED_CSV = "CHEMBL217_filtered.csv"
DATA_CURATED_CSV = "CHEMBL217_curated.csv"
DATA_FINAL_CSV = "CHEMBL217_final.csv"

In [18]:
# Part1: Data Retrieval from ChEMBL (Optimized)

def retrieve_data(target_id='CHEMBL217', endpoints=["Ki"], relations=["="]):
    
    activity = new_client.activity
    
    # Filter directly at the API level for faster retrieval
    res = activity.filter(
        target_chembl_id=target_id,
        standard_type__in=endpoints,
        relation__in=relations
        ).only([
        'canonical_smiles',
        'standard_type',
        'standard_value',
        'standard_units',
        'molecule_chembl_id',
        'document_chembl_id',
        'data_validity_comment',
        'activity_comment'
    ])
    
    print(f"Fetching data from ChEMBL for target {target_id}...")
    
    # Convert directly to list (single API call with filters applied server-side)
    res_list = list(res)
    print(f"Total filtered activities found: {len(res_list)}")
    
    # Use list comprehension for faster DataFrame creation
    data_list = [
        {
            'smiles': entry.get("canonical_smiles"),
            'standard_type': entry.get('standard_type'),
            'value': entry.get("standard_value"),
            'units': entry.get("standard_units"),
            'chembl_id': entry.get("molecule_chembl_id"),
            'document_chembl_id': entry.get("document_chembl_id"),
            'data_validity_comment': entry.get("data_validity_comment"),
            'activity_comment': entry.get("activity_comment"),
            'warning_flag': entry.get("data_validity_comment") is not None
        }
        for entry in res_list
    ]
    
    df = pd.DataFrame(data_list)
    print(f"\n✓ {len(df)} molecules collected with {endpoints} endpoints.")
    
    return df

df_raw = retrieve_data()

Fetching data from ChEMBL for target CHEMBL217...
Total filtered activities found: 11961

✓ 11961 molecules collected with ['Ki'] endpoints.


In [19]:
# Show columns and a preview
print("\nraw_df.columns:", list(df_raw.columns))
display(df_raw.head(10))
# Save
df_raw.to_csv(RAW_ALL_CSV, index=False)
print("\nSaved raw to:", RAW_ALL_CSV)


raw_df.columns: ['smiles', 'standard_type', 'value', 'units', 'chembl_id', 'document_chembl_id', 'data_validity_comment', 'activity_comment', 'warning_flag']


,smiles,standard_type,value,units,chembl_id,document_chembl_id,data_validity_comment,activity_comment,warning_flag
0,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,0.067,nM,CHEMBL156651,CHEMBL1132286,None,None,False
1,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,190.0,nM,CHEMBL156651,CHEMBL1132286,None,None,False
2,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,0.196,nM,CHEMBL156651,CHEMBL1132286,None,None,False
3,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,260.0,nM,CHEMBL156651,CHEMBL1132286,None,None,False
4,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,0.05,nM,CHEMBL156651,CHEMBL1132286,None,None,False
5,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,188.0,nM,CHEMBL156651,CHEMBL1132286,None,None,False
6,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,0.09,nM,CHEMBL156651,CHEMBL1132286,None,None,False
7,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,211.0,nM,CHEMBL156651,CHEMBL1132286,None,None,False
8,CC1Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)CC1)CC3,Ki,139.0,nM,CHEMBL349833,CHEMBL1136593,None,None,False
9,CC1(C)Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)C...,Ki,201.0,nM,CHEMBL439646,CHEMBL1136593,None,None,False



Saved raw to: CHEMBL217_raw.csv


In [20]:
#Part 2: Structure preparation (sanitize/neutralize), call data in, show detections, save
def _InitialiseNeutralisationReactions():
    patts = (
        # Imidazoles
        ('[n+;H]', 'n'),
        # Amines
        ('[N+;!H0]', 'N'),
        # Carboxylic acids and alcohols
        ('[$([O-]);!$([O-][#7])]', 'O'),
        # Thiols
        ('[S-;X1]', 'S'),
        # Sulfonamides
        ('[$([N-;X2]S(=O)=O)]', 'N'),
        # Enamines
        ('[$([N-;X2][C,N]=C)]', 'N'),
        # Tetrazoles
        ('[n-]', '[nH]'),
        # Sulfoxides
        ('[$([S-]=O)]', 'S'),
        # Amides
        ('[$([N-]C=O)]', 'N'),
    )
    return [(Chem.MolFromSmarts(x), Chem.MolFromSmiles(y, False)) for x, y in patts]

def sanitize_mol(smiles: str):
    if smiles is None or not isinstance(smiles, str):
        return smiles, True
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return smiles, True
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return smiles, True
    return smiles, False

def neutralize_mol(smiles: str):
    neutralized = False
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return smiles, False
    for reactant, product in _InitialiseNeutralisationReactions():
        while mol.HasSubstructMatch(reactant):
            neutralized = True
            rms = AllChem.ReplaceSubstructs(mol, reactant, product)
            mol = rms[0]
    smiles_out = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    return smiles_out, neutralized

def prepare_structures(smiles: str, remove_salts=True, do_sanitize=True, do_neutralize=True):
    salt = False
    failed = False
    neutralized = False

    if smiles is None or not isinstance(smiles, str) or not smiles.strip():
        return 'missing', False, True, False

    if remove_salts:
        salt = ("." in smiles)

    if do_sanitize:
        smiles, failed = sanitize_mol(smiles)

    if do_neutralize and not failed:
        smiles, neutralized = neutralize_mol(smiles)

    return smiles, salt, failed, neutralized

# Load raw data from previous cell’s save
df_raw = pd.read_csv(RAW_ALL_CSV)

# Apply structure prep
prep = df_raw['smiles'].apply(lambda s: prepare_structures(s, True, True, True))
df_raw[['CuratedSmiles', 'IsSalt', 'FailedSanit', 'Neutralized']] = pd.DataFrame(prep.tolist(), index=df_raw.index)

# Analysis: what was detected at this step
print("Structure prep analysis:")
total = len(df_raw)
print(" - Input rows:", total)
print(" - IsSalt True:", int(df_raw['IsSalt'].sum()))
print(" - FailedSanit True:", int(df_raw['FailedSanit'].sum()))
print(" - Neutralized True:", int(df_raw['Neutralized'].sum()))
print(" - warning_flag True:", int(df_raw['warning_flag'].sum()))
print(" - Unique CuratedSmiles:", df_raw['CuratedSmiles'].nunique())

# Now filter (remove salts, failed, and warning_flag)
f1 = df_raw.loc[~df_raw['IsSalt']].copy()
f2 = f1.loc[~f1['FailedSanit']].copy()
struct_df = f2.loc[~f2['warning_flag']].copy()

# Duplicates overview AFTER filtering
dup_counts = struct_df['CuratedSmiles'].value_counts()
dup_groups = (dup_counts > 1).sum()
dup_extra_rows = int(dup_counts[dup_counts > 1].sum() - dup_groups)

print("\nPost-filter analysis:")
print(" - Remaining rows:", len(struct_df))
print(" - Removed salts:", total - len(f1))
print(" - Removed failed sanitization:", len(f1) - len(f2))
print(" - Removed warning_flag:", len(f2) - len(struct_df))
print(" - Duplicate groups:", int(dup_groups))
print(" - Extra duplicate rows:", int(dup_extra_rows))

print("\nstruct_df.columns:", list(struct_df.columns))
display(struct_df.head(10))

# Save
# struct_df.to_csv(RAW_STRUCT_FILTERED_CSV, index=False)
print("\nSaved structure-filtered to:", RAW_STRUCT_FILTERED_CSV)
struct_df.to_csv(RAW_STRUCT_FILTERED_CSV, index=False)


Structure prep analysis:
 - Input rows: 11961
 - IsSalt True: 640
 - FailedSanit True: 22
 - Neutralized True: 43
 - warning_flag True: 28
 - Unique CuratedSmiles: 8433

Post-filter analysis:
 - Remaining rows: 11272
 - Removed salts: 640
 - Removed failed sanitization: 22
 - Removed warning_flag: 27
 - Duplicate groups: 2228
 - Extra duplicate rows: 3390

struct_df.columns: ['smiles', 'standard_type', 'value', 'units', 'chembl_id', 'document_chembl_id', 'data_validity_comment', 'activity_comment', 'warning_flag', 'CuratedSmiles', 'IsSalt', 'FailedSanit', 'Neutralized']


,smiles,standard_type,value,units,chembl_id,document_chembl_id,data_validity_comment,activity_comment,warning_flag,CuratedSmiles,IsSalt,FailedSanit,Neutralized
0,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,0.067,nM,CHEMBL156651,CHEMBL1132286,NaN,NaN,False,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,False,False,False
1,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,190.000,nM,CHEMBL156651,CHEMBL1132286,NaN,NaN,False,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,False,False,False
2,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,0.196,nM,CHEMBL156651,CHEMBL1132286,NaN,NaN,False,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,False,False,False
3,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,260.000,nM,CHEMBL156651,CHEMBL1132286,NaN,NaN,False,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,False,False,False
4,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,0.050,nM,CHEMBL156651,CHEMBL1132286,NaN,NaN,False,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,False,False,False
5,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,188.000,nM,CHEMBL156651,CHEMBL1132286,NaN,NaN,False,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,False,False,False
6,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,0.090,nM,CHEMBL156651,CHEMBL1132286,NaN,NaN,False,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,False,False,False
7,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,Ki,211.000,nM,CHEMBL156651,CHEMBL1132286,NaN,NaN,False,NC(=O)[C@H]1CS[C@@H]2CC[C@]3(CCCN3C(=O)[C@@H]3...,False,False,False
8,CC1Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)CC1)CC3,Ki,139.000,nM,CHEMBL349833,CHEMBL1136593,NaN,NaN,False,CC1Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)CC1)CC3,False,False,False
9,CC1(C)Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)C...,Ki,201.000,nM,CHEMBL439646,CHEMBL1136593,NaN,NaN,False,CC1(C)Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)C...,False,False,False



Saved structure-filtered to: CHEMBL217_filtered.csv


In [21]:
#Part 4: Experimental curation (Dixon’s Q, stats), call data inside, show detections, save
# JUPYTER CELL 4: Load struct → outlier filtering + stats → analysis → save

def _InitLookUp(alpha=0.05):
    if alpha == 0.10:
        q_tab = [0.941, 0.765, 0.642, 0.56, 0.507, 0.468, 0.437, 0.412, 0.392, 0.376, 0.361, 0.349, 0.338, 0.329,
                 0.32, 0.313, 0.306, 0.3, 0.295, 0.29, 0.285, 0.281, 0.277, 0.273, 0.269, 0.266, 0.263, 0.26]
    elif alpha == 0.05:
        q_tab = [0.97, 0.829, 0.71, 0.625, 0.568, 0.526, 0.493, 0.466, 0.444, 0.426, 0.41, 0.396, 0.384, 0.374,
                 0.365, 0.356, 0.349, 0.342, 0.337, 0.331, 0.326, 0.321, 0.317, 0.312, 0.308, 0.305, 0.301, 0.29]
    elif alpha == 0.01:
        q_tab = [0.994, 0.926, 0.821, 0.74, 0.68, 0.634, 0.598, 0.568, 0.542, 0.522, 0.503, 0.488, 0.475, 0.463,
                 0.452, 0.442, 0.433, 0.425, 0.418, 0.411, 0.404, 0.399, 0.393, 0.388, 0.384, 0.38, 0.376, 0.372]
    else:
        q_tab = []
    return {n: q for n, q in zip(range(3, len(q_tab)+1), q_tab)}

def dixon_test(data, left=True, right=True, alpha=0.05):
    q_dict = _InitLookUp(alpha=alpha)
    if not (3 <= len(data) <= max(q_dict.keys(), default=0)):
        return []
    sdata = sorted(data)
    Q_mindiff, Q_maxdiff = (0, 0), (0, 0)
    if left:
        Q_min = (sdata[1] - sdata[0])
        try:
            Q_min /= (sdata[-1] - sdata[0])
            Q_mindiff = (Q_min - q_dict[len(data)], sdata[0])
        except ZeroDivisionError:
            pass
    if right:
        Q_max = abs((sdata[-2] - sdata[-1]))
        try:
            Q_max /= abs((sdata[0] - sdata[-1]))
            Q_maxdiff = (Q_max - q_dict[len(data)], sdata[-1])
        except ZeroDivisionError:
            pass

    if not Q_mindiff[0] > 0 and not Q_maxdiff[0] > 0:
        return []
    if Q_mindiff[0] == Q_maxdiff[0]:
        return [Q_mindiff[1], Q_maxdiff[1]]
    if Q_mindiff[0] > Q_maxdiff[0]:
        return [Q_mindiff[1]]
    return [Q_maxdiff[1]]

def compute_stats(values: np.ndarray):
    n = len(values)
    mean = float(np.mean(values)) if n else np.nan
    std = float(np.std(values, ddof=0)) if n else np.nan
    sem = float(std/np.sqrt(n)) if n else np.nan
    return mean, std, sem, n

def concatenate_string_info(data: pd.DataFrame, outlier_idx):
    st = data["standard_type"]
    ddoc = data["document_chembl_id"]
    mid = data["chembl_id"]
    if len(outlier_idx) > 0:
        st = st.drop(st.index[outlier_idx], errors='ignore')
        ddoc = ddoc.drop(ddoc.index[outlier_idx], errors='ignore')
        mid = mid.drop(mid.index[outlier_idx], errors='ignore')
    standard_types = ",".join(sorted(set(st.astype(str))))
    chembl_docs_ids = ",".join(sorted(set(ddoc.astype(str))))
    chembl_ids = ",".join(sorted(set(mid.astype(str))))
    return chembl_ids, chembl_docs_ids, standard_types

def curate_values(values: np.ndarray, outlier_test=True, alpha=0.05, convert_to_log=False):
    if convert_to_log:
        values = -np.log10(values / 1e9)  # p-scale from nM
    new_values = values.copy()
    outlier_idx = []
    num_outliers = 0
    if outlier_test:
        outs = dixon_test(list(values), True, True, alpha=alpha)
        if len(outs) > 0:
            mask = np.isin(values, np.array(outs))
            outlier_idx = list(np.where(mask)[0])
            new_values = values[~mask]
            num_outliers = int(mask.sum())
    mean, std, sem, n = compute_stats(new_values)
    return mean, std, sem, num_outliers, n, new_values, outlier_idx

# Load structure-filtered data
struct_df = pd.read_csv(RAW_STRUCT_FILTERED_CSV)

# Prepare numeric values
work = struct_df.copy()
work['value'] = pd.to_numeric(work['value'], errors='coerce')
work = work.dropna(subset=['CuratedSmiles', 'value'])

unique_smiles = work['CuratedSmiles'].unique()
rows = []
total_outliers = 0
groups_with_outliers = 0

for smi in unique_smiles:
    data = work.loc[work['CuratedSmiles'] == smi]
    vals = data['value'].values.astype(float)
    mean, std, sem, n_out, n_vals, new_vals, out_idx = curate_values(
        vals, outlier_test=True, alpha=0.05, convert_to_log=CONVERT_TO_LOG
    )
    if n_out > 0:
        groups_with_outliers += 1
        total_outliers += n_out
    chembl_ids, chembl_docs_ids, standard_types = concatenate_string_info(data, out_idx)
    rows.append({
        'smiles': smi,
        'exp_mean': mean,
        'exp_std': std,
        'exp_sem': sem,
        'no.entries': int(len(vals)),
        'no.outliers': int(n_out),
        'standard_types': standard_types,
        'chembl_id': chembl_ids,
        'chembl_id_doc': chembl_docs_ids
    })

exp_df = pd.DataFrame(rows)

# Remove high-std groups (mimic log10(std) >= 1 => std >= 10)
before_groups = len(exp_df)
if RM_HIGH_STD:
    exp_df = exp_df.loc[~(exp_df['exp_std'] >= 10)].copy()
removed_high_std = before_groups - len(exp_df)

# Analysis
print("Experimental curation analysis:")
print(" - Input groups (unique CuratedSmiles):", len(unique_smiles))
print(" - Groups with outliers:", groups_with_outliers)
print(" - Total outliers removed:", total_outliers)
print(" - Removed high-std groups (std >= 10):", removed_high_std)
print(" - Final groups:", len(exp_df))

print("\nexp_df.columns:", list(exp_df.columns))
display(exp_df.head(10))

# Save
exp_df.to_csv(DATA_CURATED_CSV, index=False)
print("\nSaved curated to:", DATA_CURATED_CSV)


C:\Users\DELL\AppData\Local\Temp\ipykernel_8140\2357716252.py:27: RuntimeWarning: invalid value encountered in scalar divide
  Q_min /= (sdata[-1] - sdata[0])
C:\Users\DELL\AppData\Local\Temp\ipykernel_8140\2357716252.py:34: RuntimeWarning: invalid value encountered in scalar divide
  Q_max /= abs((sdata[0] - sdata[-1]))
C:\Users\DELL\AppData\Local\Temp\ipykernel_8140\2357716252.py:27: RuntimeWarning: invalid value encountered in scalar divide
  Q_min /= (sdata[-1] - sdata[0])
C:\Users\DELL\AppData\Local\Temp\ipykernel_8140\2357716252.py:34: RuntimeWarning: invalid value encountered in scalar divide
  Q_max /= abs((sdata[0] - sdata[-1]))
C:\Users\DELL\AppData\Local\Temp\ipykernel_8140\2357716252.py:27: RuntimeWarning: invalid value encountered in scalar divide
  Q_min /= (sdata[-1] - sdata[0])
C:\Users\DELL\AppData\Local\Temp\ipykernel_8140\2357716252.py:34: RuntimeWarning: invalid value encountered in scalar divide
  Q_max /= abs((sdata[0] - sdata[-1]))
C:\Users\DELL\AppData\Local\Tem

Experimental curation analysis:
 - Input groups (unique CuratedSmiles): 7882
 - Groups with outliers: 172
 - Total outliers removed: 172
 - Removed high-std groups (std >= 10): 833
 - Final groups: 7049

exp_df.columns: ['smiles', 'exp_mean', 'exp_std', 'exp_sem', 'no.entries', 'no.outliers', 'standard_types', 'chembl_id', 'chembl_id_doc']


,smiles,exp_mean,exp_std,exp_sem,no.entries,no.outliers,standard_types,chembl_id,chembl_id_doc
1,CC1Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)CC1)CC3,139.0,0.0,0.0,1,0,Ki,CHEMBL349833,CHEMBL1136593
2,CC1(C)Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)C...,201.0,0.0,0.0,1,0,Ki,CHEMBL439646,CHEMBL1136593
3,Nc1cccc(-c2ccc(CCN3CCN(c4cccc5cccnc45)CC3)cc2)n1,11.0,0.0,0.0,1,0,Ki,CHEMBL87187,CHEMBL1131962
4,Cc1ccc(CN2CCN(C3CCc4cccc5c4N(CC5)C3=O)CC2)cc1,209.0,0.0,0.0,2,0,Ki,CHEMBL351963,"CHEMBL1136593,CHEMBL5233263"
5,O=C(NCCCN1CCN(c2cccc(Cl)c2Cl)CC1)c1cccc2c1-c1c...,512.0,0.0,0.0,1,0,Ki,CHEMBL309118,CHEMBL1134599
6,Oc1nc2c(N3CCN(Cc4ccccc4)CC3)cccc2[nH]1,87.3,0.0,0.0,1,0,Ki,CHEMBL315864,CHEMBL1130595
7,O=C(NCCCN1CCN(c2ccccc2)CC1)c1cccc2c1-c1ccccc1C2=O,777.0,0.0,0.0,1,0,Ki,CHEMBL322179,CHEMBL1134599
8,O=C(NCCCCN1CCN(c2cccc(Cl)c2Cl)CC1)c1ccc2c(c1)-...,262.0,0.0,0.0,2,0,Ki,CHEMBL419505,"CHEMBL1130602,CHEMBL1134599"
9,O=C(NCCCN1CCN(c2cccc(Cl)c2Cl)CC1)c1cccc2c1Cc1c...,1680.0,0.0,0.0,1,0,Ki,CHEMBL108527,CHEMBL1134599
10,O=C(NCCCCN1CCN(c2cccc(Cl)c2Cl)CC1)c1ccc2c(c1)-...,217.0,0.0,0.0,1,0,Ki,CHEMBL110411,CHEMBL1134599



Saved curated to: CHEMBL217_curated.csv


In [22]:
#Part 5 Class assignment, call curated data inside, show results, save

def class_vector(table_final: pd.DataFrame, thr_class: float):
    out = pd.DataFrame({
        'smiles': table_final['smiles'],
        'chembl_id': table_final['chembl_id'],
        'exp_mean [nM]': table_final['exp_mean'],
    })
    out['class'] = table_final['exp_mean'] < thr_class
    return out

# Load curated
exp_df = pd.read_csv(DATA_CURATED_CSV)

final_df = class_vector(exp_df, THR_CLASS)

# Analysis
n = len(final_df)
n_active = int(final_df['class'].sum())
print("Class assignment analysis:")
print(" - Total compounds:", n)
print(" - Active (exp_mean < {} nM):".format(THR_CLASS), n_active)
print(" - Inactive:", n - n_active)
print(" - % Active:", round(100.0 * n_active / n, 2) if n > 0 else 0)

print("\nfinal_df.columns:", list(final_df.columns))
display(final_df.head(10))

# Save
final_df.to_csv(DATA_FINAL_CSV, index=False)
print("\nSaved final to:", DATA_FINAL_CSV)

Class assignment analysis:
 - Total compounds: 7049
 - Active (exp_mean < 100 nM): 2721
 - Inactive: 4328
 - % Active: 38.6

final_df.columns: ['smiles', 'chembl_id', 'exp_mean [nM]', 'class']


,smiles,chembl_id,exp_mean [nM],class
0,CC1Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)CC1)CC3,CHEMBL349833,139.0,False
1,CC1(C)Cc2cccc3c2N1C(=O)C(N1CCN(Cc2ccc(Cl)cc2)C...,CHEMBL439646,201.0,False
2,Nc1cccc(-c2ccc(CCN3CCN(c4cccc5cccnc45)CC3)cc2)n1,CHEMBL87187,11.0,True
3,Cc1ccc(CN2CCN(C3CCc4cccc5c4N(CC5)C3=O)CC2)cc1,CHEMBL351963,209.0,False
4,O=C(NCCCN1CCN(c2cccc(Cl)c2Cl)CC1)c1cccc2c1-c1c...,CHEMBL309118,512.0,False
5,Oc1nc2c(N3CCN(Cc4ccccc4)CC3)cccc2[nH]1,CHEMBL315864,87.3,True
6,O=C(NCCCN1CCN(c2ccccc2)CC1)c1cccc2c1-c1ccccc1C2=O,CHEMBL322179,777.0,False
7,O=C(NCCCCN1CCN(c2cccc(Cl)c2Cl)CC1)c1ccc2c(c1)-...,CHEMBL419505,262.0,False
8,O=C(NCCCN1CCN(c2cccc(Cl)c2Cl)CC1)c1cccc2c1Cc1c...,CHEMBL108527,1680.0,False
9,O=C(NCCCCN1CCN(c2cccc(Cl)c2Cl)CC1)c1ccc2c(c1)-...,CHEMBL110411,217.0,False



Saved final to: CHEMBL217_final.csv
